# Project 13 — Visual Tracking and Analysis Laboratory

This notebook executes the first real Project 13 technical test using `tracking_test_01.mp4`.

**Run order:** Runtime → Change runtime type → **T4 GPU**, then run cells from top to bottom.


In [ ]:
import os, sys, subprocess, pathlib
print('Python:', sys.version)
subprocess.run(['nvidia-smi'])


## 1. Clone the repository


In [ ]:
REPO_URL = 'https://github.com/Peyman-mxli/SAM3-Learning-Journey.git'
REPO_DIR = '/content/SAM3-Learning-Journey'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
else:
    subprocess.run(['git','-C',REPO_DIR,'pull'], check=True)
PROJECT_DIR = os.path.join(REPO_DIR, '05-projects/13-Visual-Tracking-and-Analysis-Lab')
os.chdir(PROJECT_DIR)
print('Project:', PROJECT_DIR)


## 2. Install standard dependencies


In [ ]:
!pip -q install -r requirements.txt


## 3. Verify the real test video


In [ ]:
import cv2, os
VIDEO = 'data/input/tracking_test_01.mp4'
assert os.path.exists(VIDEO), VIDEO
cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
duration = frames / fps if fps else 0
print({'frames':frames,'fps':fps,'width':width,'height':height,'duration_seconds':duration})


## 4. Run YOLO + ByteTrack + SQLite

This produces the first Project 13 session without claiming SAM 3 results yet.


In [ ]:
!rm -f data/project13.sqlite3
!python src/pipeline.py data/input/tracking_test_01.mp4 --db data/project13.sqlite3 --confidence 0.35 --notes "Project 13 technical test 01 — YOLO + ByteTrack"


## 5. Inspect the generated Project 13 session


In [ ]:
import sqlite3, pandas as pd
conn = sqlite3.connect('data/project13.sqlite3')
sessions = pd.read_sql_query('SELECT * FROM sessions ORDER BY created_at DESC', conn)
observations = pd.read_sql_query('SELECT * FROM observations ORDER BY frame_index', conn)
display(sessions)
print('Observations:', len(observations))
print('Unique tracker IDs:', observations['tracker_id'].dropna().nunique())
print('Average confidence:', observations['confidence'].mean())
display(observations.head())
SESSION_ID = sessions.iloc[0]['session_id']
conn.close()
print('SESSION_ID =', SESSION_ID)


## 6. Export Project 13 CSV evidence


In [ ]:
!python src/export_results.py $SESSION_ID --db data/project13.sqlite3 --output-dir results
!ls -lh results


## 7. Optional SAM 3 run

Mount Google Drive and set the local SAM 3 repository and checkpoint paths. The cell only runs when both paths exist.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Edit these two paths if your SAM 3 installation uses different names.
SAM3_REPO = '/content/drive/MyDrive/SAM3'
SAM3_CHECKPOINT = '/content/drive/MyDrive/SAM3/checkpoints/sam3.pt'

print('SAM3_REPO exists:', os.path.exists(SAM3_REPO))
print('SAM3_CHECKPOINT exists:', os.path.exists(SAM3_CHECKPOINT))
if os.path.exists(SAM3_REPO) and SAM3_REPO not in sys.path:
    sys.path.insert(0, SAM3_REPO)


In [ ]:
if os.path.exists(SAM3_REPO) and os.path.exists(SAM3_CHECKPOINT):
    !rm -f data/project13_sam3.sqlite3
    !python src/pipeline.py data/input/tracking_test_01.mp4 --db data/project13_sam3.sqlite3 --confidence 0.35 --sam-checkpoint "$SAM3_CHECKPOINT" --sam-prompt "person" --sam-every 10 --notes "Project 13 technical test 01 — YOLO + ByteTrack + SAM 3"
else:
    print('SAM 3 run skipped: update SAM3_REPO and SAM3_CHECKPOINT to your actual Drive paths.')


## 8. Create a machine-readable run summary


In [ ]:
import json, datetime, pathlib
conn = sqlite3.connect('data/project13.sqlite3')
obs = pd.read_sql_query('SELECT * FROM observations', conn)
ses = pd.read_sql_query('SELECT * FROM sessions', conn)
conn.close()
summary = {
    'project': '13-Visual-Tracking-and-Analysis-Lab',
    'input_video': VIDEO,
    'video_frames': frames,
    'video_fps': fps,
    'video_width': width,
    'video_height': height,
    'video_duration_seconds': duration,
    'session_id': str(ses.iloc[-1]['session_id']) if len(ses) else None,
    'observations': int(len(obs)),
    'unique_tracker_ids': int(obs['tracker_id'].dropna().nunique()) if len(obs) else 0,
    'average_confidence': float(obs['confidence'].mean()) if len(obs) else None,
    'generated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'sam3_evidence_in_baseline_run': False
}
pathlib.Path('results/run_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))


## 9. Launch the Streamlit dashboard


In [ ]:
!streamlit run app.py --server.port 8501 >/tmp/project13_streamlit.log 2>&1 &
print('Streamlit started on port 8501. Use your preferred Colab tunnel method to expose it.')


## 10. Files to preserve after the run

- `data/project13.sqlite3`
- `results/<session>_observations.csv`
- `results/<session>_tracker_summary.csv`
- `results/run_summary.json`
- SAM 3 database/results if the optional SAM 3 section succeeds

Do not mark Precision/Recall or IoU/Dice complete until ground truth is added and the evaluation scripts are executed.
